# 03 — Spotify Pipeline

Load the trained emotion classifier, map predicted emotions to Spotify genre seeds,
and fetch track recommendations. The final evaluation cell runs 10 diverse mood sentences
end-to-end so you can sanity-check the whole system.

**Before running this notebook:**
1. Copy `.env.example` to `.env` in this folder.
2. Fill in your `SPOTIPY_CLIENT_ID` and `SPOTIPY_CLIENT_SECRET`.
3. Authentication uses the Client Credentials flow — no browser login required.

In [ ]:
import os
import re
import json
import warnings
from pathlib import Path

import joblib
import pandas as pd
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

warnings.filterwarnings("ignore")

MODELS_DIR = Path("models")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
load_dotenv()  # reads .env from the current working directory

client_id     = os.getenv("SPOTIPY_CLIENT_ID")
client_secret = os.getenv("SPOTIPY_CLIENT_SECRET")

if not client_id or not client_secret:
    raise EnvironmentError(
        "Spotify credentials not found. Copy .env.example to .env and fill in your keys."
    )

auth_manager = SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
)
sp = spotipy.Spotify(auth_manager=auth_manager)
print("Spotify authenticated successfully.")

In [ ]:
vectorizer  = joblib.load(MODELS_DIR / "tfidf_vectorizer.pkl")
classifier  = joblib.load(MODELS_DIR / "emotion_classifier.pkl")

with open(MODELS_DIR / "model_meta.json") as f:
    meta = json.load(f)

print(f"Model: {meta['model']}  |  Val weighted F1: {meta['val_f1']}")
print(f"Classes: {meta['classes']}")

In [ ]:
# Spotify genre seeds mapped per emotion — all seeds are confirmed valid for the /recommendations endpoint.
# Ordered by 'vibe strength': first seed is the primary identifier, rest add colour.
EMOTION_TO_GENRES = {
    "joy":        ["pop",              "happy",             "dance"],
    "sadness":    ["sad",              "acoustic",          "singer-songwriter"],
    "anger":      ["metal",            "hard-rock",         "punk"],
    "fear":       ["ambient",          "piano",             "goth"],
    "excitement": ["edm",              "party",             "work-out"],
    "disgust":    ["grunge",           "emo",               "alt-rock"],
    "love":       ["romance",          "r-n-b",             "soul"],
    "admiration": ["classical",        "opera",             "new-age"],
    "gratitude":  ["gospel",           "folk",              "acoustic"],
    "curiosity":  ["jazz",             "world-music",       "indie"],
    "optimism":   ["indie-pop",        "reggae",            "afrobeat"],
    "neutral":    ["study",            "chill",             "piano"],
}

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def predict_emotion(text: str) -> dict:
    """Return predicted emotion and class probabilities for an input string."""
    cleaned = clean_text(text)
    vec     = vectorizer.transform([cleaned])
    emotion = classifier.predict(vec)[0]
    proba   = classifier.predict_proba(vec)[0]
    classes = classifier.classes_
    return {
        "emotion": emotion,
        "confidence": round(float(proba.max()), 3),
        "top3": sorted(zip(classes, proba.tolist()), key=lambda x: -x[1])[:3],
    }


# quick sanity check
sample = predict_emotion("It's raining and I feel melancholy but hopeful")
print(sample)

In [ ]:
def get_tracks(genre_seeds: list, limit: int = 15) -> list:
    """
    Fetch tracks using the Recommendations endpoint (genre seeds).
    Falls back to keyword search if the endpoint is restricted for this app.
    """
    try:
        resp = sp.recommendations(seed_genres=genre_seeds[:3], limit=limit)
        return resp["tracks"]
    except spotipy.SpotifyException:
        # recommendations restricted on newer apps — search by primary genre instead
        q = f"genre:{genre_seeds[0]}"
        resp = sp.search(q=q, type="track", limit=limit)
        return resp["tracks"]["items"]


def generate_playlist(mood_text: str, limit: int = 10) -> pd.DataFrame:
    """Full pipeline: text → emotion → genre seeds → track list."""
    result  = predict_emotion(mood_text)
    emotion = result["emotion"]
    genres  = EMOTION_TO_GENRES[emotion]
    tracks  = get_tracks(genres, limit=limit)

    rows = []
    for t in tracks:
        artists = ", ".join(a["name"] for a in t["artists"])
        rows.append({
            "track":   t["name"],
            "artist":  artists,
            "album":   t["album"]["name"],
            "url":     t["external_urls"]["spotify"],
        })

    playlist_df = pd.DataFrame(rows)
    return emotion, result["confidence"], genres, playlist_df

## Try It — Interactive Cell

Edit `mood_text` below and run the cell.

In [ ]:
mood_text = "It's raining and I feel melancholy but hopeful"

emotion, confidence, genres, pl = generate_playlist(mood_text)

print(f"Input   : {mood_text}")
print(f"Emotion : {emotion}  (confidence: {confidence})")
print(f"Genres  : {genres}\n")
pd.set_option("display.max_colwidth", 80)
pl

## Evaluation — 10 Diverse Mood Sentences

Run the full pipeline on each sentence and export a summary. No ground-truth labels here —
this is a subjective sanity check to see whether the emotion predictions feel right.

In [ ]:
TEST_INPUTS = [
    "I aced my exam today, absolutely over the moon!",
    "Why would anyone do something so cruel and pointless?",
    "I'm nervous about the presentation tomorrow, can't sleep.",
    "Had the most romantic dinner with her last night.",
    "Everything feels grey and pointless lately.",
    "Just got tickets to my favourite band, LETS GOOOO",
    "I wonder how butterflies know where to migrate.",
    "So grateful for the people who showed up when I needed them.",
    "Honestly disturbed by what I just watched.",
    "Things are slowly starting to look up, finally.",
]

summary_rows = []
for sentence in TEST_INPUTS:
    emotion, confidence, genres, pl = generate_playlist(sentence, limit=5)
    sample_track = pl.iloc[0]["track"] + " — " + pl.iloc[0]["artist"] if not pl.empty else "N/A"
    summary_rows.append({
        "input":        sentence[:60] + ("..." if len(sentence) > 60 else ""),
        "emotion":      emotion,
        "confidence":   confidence,
        "primary_genre": genres[0],
        "sample_track": sample_track,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "evaluation_results.csv", index=False)
summary_df

In [ ]:
# distribution of predicted emotions across our 10 test sentences — just a quick check
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3))
summary_df["emotion"].value_counts().plot(kind="bar", ax=ax, color="steelblue", edgecolor="none")
ax.set_title("Predicted emotions — 10 evaluation inputs", fontsize=12)
ax.set_xlabel("")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("outputs/eval_emotion_dist.png", bbox_inches="tight")
plt.show()

print("\nEvaluation complete. Results saved to outputs/evaluation_results.csv")